# 3. Feature Engineering

## Getting started

Welcome to the third practical session! As always, the notebook is designed to be run online directly in your browser by clicking the rocket icon on the top right and selecting `Live Code`.

## Outline

This workshop will cover the following content:

1. What is feature engineering?
1. Feature engineering in chemistry
1. Data cleaning
1. Task: Putting it all together

## What is feature engineering?

In the previous workshops we worked with datasets where the features were already numeric and ready for modelling. In practice, raw data is rarely in a form that machine learning algorithms can use directly. Feature engineering is the process of transforming raw data into informative numerical representations that a model can learn from.

Good feature engineering can make the difference between a mediocre model and an excellent one. It is often said that *feature engineering is where domain knowledge meets machine learning*. If you can craft features that capture the underlying science, even simple models can perform well.

**Why does it matter?**

Consider a simple example: predicting whether a chemical reaction will proceed at room temperature. You might have the activation energy $E_a$ and the temperature $T$ as raw inputs. A linear model trained on $E_a$ and $T$ separately may struggle, but if you engineer a feature based on the Arrhenius equation — for example, the Boltzmann factor $e^{-E_a / k_B T}$ — the problem becomes nearly trivial for a linear model.

Domain knowledge lets you encode physical relationships directly into your features, reducing the burden on the model to discover these relationships from data alone.

### A concrete example: predicting boiling points

Let's illustrate this with a toy example. Suppose we want to predict the boiling points of straight-chain alkanes from their molecular formula.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Boiling points (°C) of straight-chain alkanes C1–C10
alkanes = pd.DataFrame(
    {
        "name": [
            "Methane",
            "Ethane",
            "Propane",
            "Butane",
            "Pentane",
            "Hexane",
            "Heptane",
            "Octane",
            "Nonane",
            "Decane",
        ],
        "n_carbons": np.arange(1, 11),
        "molecular_weight": [
            16.04,
            30.07,
            44.10,
            58.12,
            72.15,
            86.18,
            100.20,
            114.23,
            128.26,
            142.29,
        ],
        "boiling_point": [
            -161.5,
            -88.6,
            -42.1,
            -0.5,
            36.1,
            68.7,
            98.4,
            125.7,
            150.8,
            174.1,
        ],
    }
)
alkanes

A first attempt might be to use the molecular weight directly as a feature.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X_mw = alkanes[["molecular_weight"]].values
y = alkanes["boiling_point"].values

model_mw = LinearRegression().fit(X_mw, y)
y_pred_mw = model_mw.predict(X_mw)

mae_mw = mean_absolute_error(y, y_pred_mw)
print(f"MAE using molecular weight alone: {mae_mw:.1f} °C")

Not bad, but the relationship between molecular weight and boiling point is not perfectly linear for small alkanes.

We know that boiling point depends on intermolecular forces, which scale roughly with surface area. For straight-chain alkanes, the surface area grows with the chain length but not linearly — the relationship is closer to a power law or logarithmic.

Let's engineer a new feature: $\log(n_\text{carbons})$.

In [ ]:
X_eng = np.column_stack(
    [
        alkanes["molecular_weight"].values,
        np.log(alkanes["n_carbons"].values),
    ]
)

model_eng = LinearRegression().fit(X_eng, y)
y_pred_eng = model_eng.predict(X_eng)

mae_eng = mean_absolute_error(y, y_pred_eng)
print(f"MAE with engineered features: {mae_eng:.1f} °C")

Let's visualise the improvement.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for ax, y_pred, title, mae in zip(
    axes,
    [y_pred_mw, y_pred_eng],
    ["Molecular weight only", "With log(n_carbons)"],
    [mae_mw, mae_eng],
):
    ax.scatter(y, y_pred)
    lims = [min(y.min(), y_pred.min()) - 10, max(y.max(), y_pred.max()) + 10]
    ax.plot(lims, lims, "k--", alpha=0.5)
    ax.set(
        xlabel="True boiling point (°C)",
        ylabel="Predicted (°C)",
        title=f"{title}\nMAE = {mae:.1f} °C",
        xlim=lims,
        ylim=lims,
    )

plt.tight_layout()
plt.show()

By adding a single feature grounded in chemical intuition, we substantially improved a simple linear model.

### General principles

When engineering features, keep these principles in mind:

- **Use domain knowledge.** If you know the underlying physics or chemistry, encode it. Ratios, logarithms, interaction terms, and physically motivated combinations often work well.
- **Keep it simple.** Start with a few well-motivated features before adding complexity. More features is not always better and can lead to overfitting, especially with small datasets.
- **Let the data guide you.** Exploratory plots and correlation analysis can reveal which raw features are informative and which transformations might help.
- **Beware of data leakage.** Never include information in your features that would not be available at prediction time.

## Feature engineering in chemistry

Chemical data presents a unique challenge for machine learning: molecules are not numbers. They are graphs of atoms and bonds, or 3D arrangements of nuclei in space. Before we can apply any ML algorithm, we need to convert molecular structures into fixed-length numerical vectors.

### Representing molecules: SMILES and 3D coordinates

The two most common ways to represent molecules in datasets are:

**SMILES** (Simplified Molecular-Input Line-Entry System) — a compact string notation that encodes the molecular graph. For example:
- Water: `O`
- Ethanol: `CCO`
- Benzene: `c1ccccc1`
- Aspirin: `CC(=O)Oc1ccccc1C(=O)O`

**XYZ coordinates** — the 3D Cartesian coordinates of each atom, often obtained from geometry optimisation calculations or crystal structures.

Neither of these representations can be fed directly into most ML algorithms (which expect fixed-length numerical input). We need to convert them into **molecular descriptors** or **fingerprints**.

### Molecular fingerprints

Molecular fingerprints are bit vectors (or count vectors) that encode structural information about a molecule. Different fingerprint types capture different aspects of molecular structure.

**Morgan fingerprints (ECFP — Extended Connectivity Fingerprints)**

Morgan fingerprints are circular fingerprints that capture the local chemical environment around each atom. The algorithm works by:
1. Assigning an initial identifier to each atom based on its properties (element, charge, etc.).
2. Iteratively updating each identifier by incorporating information from neighbouring atoms.
3. After a specified number of iterations (the *radius*), collecting all identifiers and hashing them into a fixed-length bit vector.

A radius of 2 (ECFP4) means each bit encodes a substructure within a diameter of 4 bonds. These are widely used for similarity searching and ML.

<img src="http://www.yueqian.org/images/fingerprints.png" alt="Illustrative example of Morgan fingerprint for Ibuprofen" width="400">

**MACCS keys**

MACCS (Molecular ACCess System) keys are a set of 166 predefined structural patterns. Each bit indicates the presence or absence of a specific substructure or functional group (e.g., "contains a six-membered ring", "contains an OH group"). They are easy to interpret but limited to the predefined set of patterns.

**RDKit topological fingerprints**

RDKit's built-in fingerprints enumerate all linear paths of bonds up to a specified length and hash them into a bit vector. They capture connectivity patterns along paths through the molecule.

### Using RDKit

[RDKit](https://www.rdkit.org/) is the standard open-source cheminformatics toolkit for Python. Let's use it to compute fingerprints for some simple molecules.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

# Parse SMILES strings into molecule objects
smiles_list = ["CCO", "c1ccccc1", "CC(=O)Oc1ccccc1C(=O)O", "C(=O)N"]
names = ["Ethanol", "Benzene", "Aspirin", "Formamide"]

mols = [Chem.MolFromSmiles(s) for s in smiles_list]

# Draw the molecules
Draw.MolsToGridImage(mols, molsPerRow=4, legends=names)

Now let's compute Morgan fingerprints for these molecules. We'll use a radius of 2 (ECFP4) and a fingerprint length of 1024 bits.

In [ ]:
from rdkit.Chem import AllChem
from rdkit.Chem import rdFingerprintGenerator

# Create a Morgan fingerprint generator with radius 2 and 1024 bits
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024)

# Compute Morgan fingerprints (radius=2, 1024 bits)
fps = [mfpgen.GetFingerprint(mol) for mol in mols]

# Convert to numpy arrays for easier manipulation
fp_arrays = np.array([list(fp) for fp in fps])

print(f"Fingerprint shape: {fp_arrays.shape}")
print("Number of 'on' bits per molecule:")
for name, fp in zip(names, fp_arrays):
    print(f"  {name}: {fp.sum()} / {len(fp)} bits")

Each molecule is now represented as a vector of 1024 binary values. We can visualise these fingerprints to see how different molecules produce different patterns.

In [ ]:
fig, axes = plt.subplots(len(names), 1, figsize=(10, 4), sharex=True)

for ax, name, fp in zip(axes, names, fp_arrays):
    ax.imshow(fp.reshape(1, -1), aspect="auto", cmap="Blues", interpolation="none")
    ax.set_yticks([0])
    ax.set_yticklabels([name])

axes[-1].set_xlabel("Bit index")
fig.suptitle("Morgan fingerprints (radius=2, 1024 bits)")
plt.tight_layout()
plt.show()

We can also use the Tanimoto similarity (also called the Jaccard index) to compare molecules based on their fingerprints. The Tanimoto similarity between two bit vectors $A$ and $B$ is defined as:

$$
T(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{c}{a + b - c}
$$

where $a$ and $b$ are the number of 'on' bits in $A$ and $B$ respectively, and $c$ is the number of bits that are 'on' in both.

In [ ]:
from rdkit import DataStructs

# Calculate pairwise Tanimoto similarity
n = len(fps)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = DataStructs.TanimotoSimilarity(fps[i], fps[j])

fig, ax = plt.subplots(figsize=(5, 4))
cax = ax.matshow(sim_matrix, cmap="viridis", vmin=0, vmax=1)
ax.set(xticks=range(n), yticks=range(n), xticklabels=names, yticklabels=names)
plt.xticks(rotation=45, ha="left")
plt.colorbar(cax, label="Tanimoto similarity")
plt.title("Pairwise molecular similarity")
plt.tight_layout()
plt.show()

Which pairs of molecules are most similar? Does this match your chemical intuition?

### MACCS keys

Let's also compute MACCS keys for comparison. These are a set of 166 predefined structural keys.

In [ ]:
from rdkit.Chem import MACCSkeys

maccs_fps = [MACCSkeys.GenMACCSKeys(mol) for mol in mols]
maccs_arrays = np.array([list(fp) for fp in maccs_fps])

print(f"MACCS fingerprint shape: {maccs_arrays.shape}")
print("Number of 'on' bits per molecule:")
for name, fp in zip(names, maccs_arrays):
    print(f"  {name}: {fp.sum()} / {len(fp)} bits")

MACCS keys are much shorter (167 bits) than Morgan fingerprints. Each bit corresponds to a specific, interpretable structural feature. This makes them easy to understand but less flexible than Morgan fingerprints.

### Molecular descriptors

Beyond fingerprints, RDKit can calculate a wide range of numerical molecular descriptors — continuous-valued properties like molecular weight, LogP (a measure of hydrophobicity), topological polar surface area, and many more.

In [ ]:
from rdkit.Chem import Descriptors

# Calculate some common descriptors
descriptor_data = []
for mol, name in zip(mols, names):
    descriptor_data.append(
        {
            "Molecule": name,
            "MolWt": Descriptors.MolWt(mol),
            "LogP": Descriptors.MolLogP(mol),
            "TPSA": Descriptors.TPSA(mol),
            "NumHDonors": Descriptors.NumHDonors(mol),
            "NumHAcceptors": Descriptors.NumHAcceptors(mol),
            "NumRotatableBonds": Descriptors.NumRotatableBonds(mol),
        }
    )

desc_df = pd.DataFrame(descriptor_data)
desc_df

These descriptors can be used alongside or instead of fingerprints as features for ML models. In practice, combining fingerprints with selected descriptors often gives the best results.

### Featurising a real dataset

Let's apply these techniques to the ChEMBL small molecule dataset. We'll load it, compute Morgan fingerprints from the SMILES strings, and create a feature matrix suitable for ML.

In [ ]:
# Load the ChEMBL dataset
chembl = pd.read_csv(
    "https://raw.githubusercontent.com/aichemy-hub/intro_to_ml_for_chemists/refs/heads/main/datasets/chembl-small.csv"
)

print(f"Dataset shape: {chembl.shape}")
chembl.head()

In [ ]:
# Let's look at the SMILES column — this is what we'll featurise
print("Example SMILES strings:")
for smi in chembl["Smiles"].dropna().head(5):
    print(f"  {smi}")

In [ ]:
def smiles_to_morgan(smiles):
    """Convert a SMILES string to a Morgan fingerprint.

    Uses the mfpgen generator we created earlier with radius=2 and 1024 bits.
    Returns None if the SMILES cannot be parsed.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return list(mfpgen.GetFingerprint(mol))


# Test on a single molecule
test_fp = smiles_to_morgan("CCO")
print(f"Fingerprint length: {len(test_fp)}")
print(f"First 20 bits: {test_fp[:20]}")

Now let's featurise a sample of molecules from the dataset. We'll work with a subset to keep things fast.

In [ ]:
# Take a random sample and compute fingerprints
sample = chembl.dropna(subset=["Smiles"]).sample(500, random_state=42).copy()

# Compute fingerprints (this may take a moment)
fp_list = sample["Smiles"].apply(smiles_to_morgan)

# Some SMILES may fail to parse — let's check
n_failed = fp_list.isna().sum()
print(f"Failed to parse: {n_failed} / {len(sample)} molecules")

# Remove failed molecules and convert to array
sample = sample[fp_list.notna()].copy()
fp_list = fp_list.dropna()
X_fp = np.array(fp_list.tolist())

print(f"Feature matrix shape: {X_fp.shape}")

We now have a feature matrix where each row is a molecule and each column is a bit in the Morgan fingerprint. This matrix is ready to be used as input to any scikit-learn model.

### Alternatives to RDKit

While RDKit is the most widely used toolkit, there are several alternatives worth knowing about:

- **[Mordred](https://github.com/mordred-descriptor/mordred):** a Python package that can calculate over 1800 molecular descriptors. It is particularly fast and can handle large molecules that other tools struggle with.
- **[DeepChem](https://deepchem.io/):** a full machine learning framework for chemistry that provides extensive featurisers including fingerprints, graph-based representations, and learned embeddings (Mol2Vec).
- **[scikit-fingerprints](https://github.com/scikit-fingerprints/scikit-fingerprints):** provides over 30 fingerprint implementations with a scikit-learn compatible API, making it easy to drop into existing ML pipelines.

For this course we'll stick with RDKit as it is the most widely used and is already in our environment.

## Data cleaning

Real-world chemical datasets are almost always messy. Before building any model, it is necessary to clean the data. This section covers the most common issues and how to handle them.

### Identifying missing data

The first step is always to understand what your data looks like and where the gaps are.

In [ ]:
# Let's look at the missing values in the ChEMBL dataset
missing = chembl.isnull().sum()
missing_pct = (missing / len(chembl) * 100).round(1)

missing_info = pd.DataFrame({"Missing": missing, "Percent": missing_pct})
missing_info[missing_info["Missing"] > 0].sort_values("Percent", ascending=False)

We can also visualise the pattern of missing data. This can reveal whether data is missing at random or whether there are systematic gaps.

In [ ]:
# Select numeric columns with some missing data
numeric_cols = chembl.select_dtypes(include=[np.number]).columns
cols_with_missing = [c for c in numeric_cols if chembl[c].isnull().any()]

fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(
    chembl[cols_with_missing].isnull().values[:200].T,
    aspect="auto",
    cmap="Reds",
    interpolation="none",
)
ax.set_yticks(range(len(cols_with_missing)))
ax.set_yticklabels(cols_with_missing)
ax.set_xlabel("Sample index (first 200)")
ax.set_title("Missing data pattern (red = missing)")
plt.tight_layout()
plt.show()

### Strategies for handling missing data

There are several common approaches to dealing with missing values:

**1. Drop rows with missing values**

The simplest approach. This works well when only a small fraction of data is missing, but can lead to significant data loss otherwise.

In [ ]:
# Example: drop rows where AlogP is missing
chembl_subset = chembl[["AlogP", "Molecular Weight", "PSA", "HBA", "HBD"]].copy()

print(f"Original size: {len(chembl_subset)}")
chembl_clean = chembl_subset.dropna(subset=["AlogP"])
print(f"After dropping rows missing AlogP: {len(chembl_clean)}")

**2. Drop columns with too much missing data**

If a feature is missing for most samples, it may not be informative enough to keep.

In [ ]:
# Drop columns where more than 50% of values are missing
threshold = 0.5
cols_to_keep = [c for c in chembl.columns if chembl[c].isnull().mean() < threshold]
print(f"Keeping {len(cols_to_keep)} of {len(chembl.columns)} columns")

**3. Imputation — filling in missing values**

Instead of dropping data, we can fill in missing values with a reasonable estimate. Common strategies include using the mean, median, or most frequent value of the column. Scikit-learn provides a convenient `SimpleImputer` class for this.

In [ ]:
from sklearn.impute import SimpleImputer

# Create a small example with some missing values
example = chembl[["AlogP", "PSA", "HBA"]][15:30].copy()
print("Before imputation:")
print(example.head(30))
print(f"\nMissing values: {example.isnull().sum().sum()}")

# Impute missing values with the column median
imputer = SimpleImputer(strategy="median")
example_imputed = pd.DataFrame(
    imputer.fit_transform(example),
    columns=example.columns,
    index=example.index,
)
print("\nAfter median imputation:")
print(example_imputed.head(15))
print(f"\nMissing values: {example_imputed.isnull().sum().sum()}")

**Which strategy should you use?** It depends on the context:
- If you have lots of data and only a few rows have missing values, dropping is fine.
- If a column has very high missingness (say >50%), consider dropping the column.
- Imputation is useful when you can't afford to lose data, but be aware that imputed values are estimates and can introduce bias.

### Handling outliers

Outliers are data points that are unusually far from the rest of the data. They can arise from measurement errors, data entry mistakes, or genuinely unusual samples.

Let's look at the distribution of molecular weight in the ChEMBL dataset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

mw = chembl["Molecular Weight"].dropna()

axes[0].hist(mw, bins=50, edgecolor="black")
axes[0].set(
    xlabel="Molecular Weight", ylabel="Count", title="Distribution of molecular weight"
)

axes[1].boxplot(mw, vert=True)
axes[1].set(ylabel="Molecular Weight", title="Box plot")

plt.tight_layout()
plt.show()

print(f"Mean: {mw.mean():.1f}")
print(f"Median: {mw.median():.1f}")
print(f"Std: {mw.std():.1f}")
print(f"Min: {mw.min():.1f}, Max: {mw.max():.1f}")

There are a few common approaches to detecting outliers:

**Z-score method**: flag points more than a certain number of standard deviations from the mean. A common threshold is 3.

In [ ]:
# Z-score outlier detection
z_scores = (mw - mw.mean()) / mw.std()
outliers_z = mw[np.abs(z_scores) > 3]
print(f"Z-score outliers (|z| > 3): {len(outliers_z)} molecules")

**IQR (Interquartile Range) method**: flag points below $Q_1 - 1.5 \times \text{IQR}$ or above $Q_3 + 1.5 \times \text{IQR}$, where $Q_1$ and $Q_3$ are the 25th and 75th percentiles. This is the method used by box plots.

In [ ]:
# IQR outlier detection
Q1 = mw.quantile(0.25)
Q3 = mw.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_iqr = mw[(mw < lower) | (mw > upper)]
print(f"IQR outliers: {len(outliers_iqr)} molecules")
print(f"Bounds: [{lower:.1f}, {upper:.1f}]")

**What to do with outliers?**

- **Investigate first.** Are they data errors or genuine measurements? In chemistry, extreme values may represent real but unusual molecules.
- **Remove them** if they are clearly erroneous (e.g., negative molecular weights).
- **Clip (winsorise) them** to a reasonable range if you want to keep the data points but reduce their influence.
- **Use robust methods.** Some models (e.g., tree-based methods like random forests) are naturally robust to outliers. Others (e.g., linear regression) are highly sensitive.

### Feature scaling

Many ML algorithms are sensitive to the scale of input features. If one feature ranges from 0 to 1 and another from 0 to 10,000, the model may be dominated by the larger-scale feature.

Two common approaches are standardisation (zero mean, unit variance) and min-max scaling (rescaling to [0, 1]).

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Take some numeric features from the ChEMBL dataset
features = ["Molecular Weight", "AlogP", "PSA", "HBA", "HBD"]
X_raw = chembl[features].dropna()

print("Before scaling:")
print(X_raw.describe().round(2))

In [ ]:
# Standardisation (z-score normalisation)
scaler = StandardScaler()
X_standard = pd.DataFrame(
    scaler.fit_transform(X_raw), columns=features, index=X_raw.index
)

print("After standardisation:")
print(X_standard.describe().round(2))

In [ ]:
# Min-max scaling
scaler = MinMaxScaler()
X_minmax = pd.DataFrame(
    scaler.fit_transform(X_raw), columns=features, index=X_raw.index
)

print("After min-max scaling:")
print(X_minmax.describe().round(2))

**When to scale?**
- **Always scale** when using distance-based algorithms (k-nearest neighbours, SVMs, PCA) or gradient-based methods (neural networks, logistic regression).
- **No need to scale** for tree-based methods (decision trees, random forests, gradient boosted trees) as they are invariant to feature scaling.
- **Fit on training data only.** Always fit the scaler on your training set and use `transform` (not `fit_transform`) on your test set to avoid data leakage.

## Additional reading

- The [RDKit documentation](https://www.rdkit.org/docs/) is an excellent resource for molecular featurisation.
- For a comprehensive overview of molecular representations for ML, see: *Molecular representations in AI-driven drug discovery* by David et al., 2020.
- Chapter 2 of [Hands-on Machine Learning](https://www.oreilly.com/library/view/hands-on-machine-learning/9781492032632/) provides a good introduction to data cleaning and feature engineering in general.